In [ ]:
# ============================
# CÉLULA #1 | Importes Iniciais
# Protótipo PrePol — versão 0.3
# ============================

import numpy as np
import pandas as pd

# Geoespacial (usaremos H3 como discretização única no protótipo)
import geopandas as gpd
from shapely.geometry import Point
import pyproj
import h3

# Modelo e utilidades
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# Visualização rápida (diagnósticos e mapas simples)
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# ==========================================
# CÉLULA #2 | Configurações & Paths
# Protótipo PrePol — parâmetros mínimos
# ==========================================
"""
Propósito
---------
Definir caminhos de entrada/saída, parâmetros espaciais (H3) e temporais, e utilitários
de configuração para o pipeline do PrePol. Nenhuma leitura de dados é feita aqui.

Notas importantes
-----------------
- Não criamos DATA_DIR automaticamente; a ausência de arquivos brutos deve ser visível.
- H3_RES: resolução 7 é relativamente fina (~0,16 km²). Se a meta for ~1 km², considere 6.
- TIME_FREQ='W' em pandas ancora a semana no DOMINGO por padrão; ajuste para 'W-MON' se desejar.
- DEFAULT_TZ é usado apenas para parsing/coerência temporal (não fazemos conversões aqui).
"""

from pathlib import Path
import os

# ---------------------------
# Diretórios e arquivos (ajuste conforme seu ambiente local/Colab/Drive)
# ---------------------------
# Ex.: BASE_DIR = Path("/content/drive/MyDrive/PrePol")  # se necessário
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "prepol_data" / "raw"             # não será criado automaticamente
OUTPUT_DIR = BASE_DIR / "prepol_out"                    # artefatos (modelos, figuras, tabelas)

# Arquivos RDO (constantes; mantenha nomes/intervalos como anotação)
RDO_FILES = {
    "RDO_1": DATA_DIR / "RDO_1.csv",  # 2010–2012
    "RDO_2": DATA_DIR / "RDO_2.csv",  # 2013–2015
    "RDO_3": DATA_DIR / "RDO_3.csv",  # 2016–2017
}

# Criamos apenas OUTPUT_DIR (artefatos do pipeline)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------
# Discretização espacial (H3)
# ---------------------------
H3_RES = 7  # resolução H3 (7 é relativamente fina, ~0,16 km²)
# ---------------------------
# Janela temporal
# ---------------------------
# 'D' = diário (ancorada no domingo por padrão do pandas). Alternativas: 'W-MON', 'D' (diário), 'M' (mensal).
TIME_FREQ = "D"

# ---------------------------
# Colunas esperadas no dataset bruto (nomes CANÔNICOS, maiúsculos)
# ---------------------------
COL_LAT = "LATITUDE"
COL_LON = "LONGITUDE"
COL_DATETIME = "DATA_OCORRENCIA_BO"     # datetime do evento (será parseado em célula de limpeza)
COL_CRIME_TYPE = "RUBRICA"        # opcional: permitir filtragem por tipo

# ---------------------------
# Filtro opcional por tipo de crime (None = usar todos)
# ---------------------------
TARGET_CRIME_TYPE = None  # ex.: "ROUBO", "FURTO", ...

# ---------------------------
# Timezone padrão do conjunto (para coerência no parsing; não convertemos aqui)
# ---------------------------
DEFAULT_TZ = "America/Recife"

# ---------------------------
# Métricas operacionais (Precision@k espacial)
# ---------------------------
EVAL_KS = [0.05, 0.10]  # 5% e 10% do espaço (frações entre 0 e 1)

# ---------------------------
# Reprodutibilidade (fallback caso não tenha sido definido na célula 1)
# ---------------------------
try:
    RANDOM_STATE  # noqa: F821
except NameError:
    RANDOM_STATE = 42

# ---------------------------
# Utilidades
# ---------------------------
def normalize_df_columns_to_upper(df):
    """
    Normaliza nomes de colunas para UPPERCASE e sem espaços nas extremidades.

    Parâmetros
    ----------
    df : pandas.DataFrame

    Retorno
    -------
    pandas.DataFrame
        O próprio DataFrame com colunas renomeadas (efeito inplace nos nomes).
    """
    df.columns = df.columns.astype(str).str.strip().str.upper()
    return df


def maybe_filter_target_crime(df, dataset_label="dataset"):
    """
    Aplica filtro por TARGET_CRIME_TYPE se configurado e a coluna existir.

    Parâmetros
    ----------
    df : pandas.DataFrame
        DataFrame bruto já com colunas normalizadas (maiúsculas).
    dataset_label : str
        Rótulo de identificação (para logs).

    Retorno
    -------
    pandas.DataFrame
        DataFrame filtrado (ou original, se filtro não aplicável).
    """
    if TARGET_CRIME_TYPE is None:
        return df

    if COL_CRIME_TYPE not in df.columns:
        print(f"[AVISO] Coluna '{COL_CRIME_TYPE}' não encontrada em {dataset_label}; filtro ignorado.")
        return df

    target_token = str(TARGET_CRIME_TYPE).strip().upper()
    col_series = df[COL_CRIME_TYPE].astype(str).str.strip().str.upper()
    filtered = df[col_series == target_token].copy()
    removed = len(df) - len(filtered)
    print(f">>> Filtro '{TARGET_CRIME_TYPE}' em {dataset_label}: mantidas {len(filtered):,} linhas (removidas {removed:,})")
    return filtered


def _validate_config():
    """
    Validações leves de configuração para falhar cedo se algo básico estiver fora do esperado.
    """
    # EVAL_KS devem ser frações (0, 1]
    invalid = [k for k in EVAL_KS if not (0 < k <= 1)]
    if invalid:
        raise ValueError(f"EVAL_KS inválidos (devem estar em (0,1]): {invalid}")

    # Colunas canônicas não vazias
    for name, col in [("COL_LAT", COL_LAT), ("COL_LON", COL_LON), ("COL_DATETIME", COL_DATETIME)]:
        if not isinstance(col, str) or not col.strip():
            raise ValueError(f"{name} inválido: '{col}'")

    # Frequência temporal informativa
    if not isinstance(TIME_FREQ, str) or not TIME_FREQ:
        raise ValueError("TIME_FREQ inválido (string esperada).")


def check_paths():
    """
    Exibe verificação de caminhos e existência de arquivos RDO.
    Não cria DATA_DIR; apenas reporta estado.
    """
    print(">>> Verificação de caminhos")
    print(f"DATA_DIR  : {DATA_DIR.resolve()} | existe={DATA_DIR.is_dir()}")
    print(f"OUTPUT_DIR: {OUTPUT_DIR.resolve()} | existe={OUTPUT_DIR.is_dir()}")
    for k, v in RDO_FILES.items():
        print(f"{k}: {v.resolve()} | existe={v.is_file()}")


def show_config():
    """
    Exibe configuração atual do PrePol para rastreabilidade de execução.
    """
    print(">>> Configuração PrePol")
    print(f"H3_RES={H3_RES} | TIME_FREQ='{TIME_FREQ}' | TZ='{DEFAULT_TZ}'")
    print(f"Cols: lat='{COL_LAT}', lon='{COL_LON}', dt='{COL_DATETIME}', tipo='{COL_CRIME_TYPE}'")
    print(f"Filtro de crime: {TARGET_CRIME_TYPE}")
    print(f"EVAL_KS: {EVAL_KS}")
    print(f"RANDOM_STATE: {RANDOM_STATE}")


# Execução das checagens iniciais
_validate_config()
check_paths()
show_config()


In [ ]:
# ==========================================================
# CÉLULA #3 | Leitura Inicial de Dataset
# Objetivo: carregar RDO_1, RDO_2, RDO_3 e gerar relatório descritivo
# ==========================================================
"""
Lê os CSVs brutos (RDO_1/2/3) e produz um relatório descritivo leve:
- Estrutura de colunas (normalizadas para UPPERCASE).
- Tipos detectados.
- Amostra (head).
- Resumo de nulos (%) e cardinalidade (únicos) por coluna (controlável por toggle).
- Estatísticas básicas (apenas colunas numéricas).
- Resumo global de faixa temporal e tamanhos.

Sem transformações usadas posteriormente no modelo — portanto, sem risco de data leakage.

Desempenho:
- Se disponível, usa engine='pyarrow' no read_csv (alta performance).
- Caso 'pyarrow' não esteja disponível, usa engine='python' para suportar on_bad_lines='warn'.
- O toggle PROFILE_HEAVY controla o cálculo de 'nunique()' (caro em alta cardinalidade).
"""

from pathlib import Path
from typing import Dict, Optional
import pyarrow

PROFILE_HEAVY: bool = True  # define se calcula 'nunique' (custo potencialmente alto em colunas longas)


def _safe_display(obj, max_rows: int = 3) -> None:
    """
    Tenta exibir objeto via display(); se indisponível, imprime uma amostra.
    """
    try:
        display(obj)  # type: ignore[name-defined]
    except Exception:
        if hasattr(obj, "head"):
            print(obj.head(max_rows))
        else:
            print(obj)


def _choose_csv_engine() -> str:
    """
    Escolhe engine para read_csv:
    - 'pyarrow' se disponível (mais rápido; não suporta on_bad_lines).
    - Caso contrário, 'python' para permitir on_bad_lines='warn'.
    """
    if "pyarrow" in globals() and pyarrow is not None:
        return "pyarrow"
    return "python"


def carregar_dataset(path: Path, nome: str) -> Optional[pd.DataFrame]:
    """
    Lê um arquivo CSV, normaliza nomes de colunas e emite um relatório descritivo.

    Parâmetros
    ----------
    path : Path
        Caminho para o arquivo CSV.
    nome : str
        Rótulo do dataset (ex.: 'RDO_1') para logs.

    Retorno
    -------
    Optional[pandas.DataFrame]
        DataFrame carregado e inspecionado, ou None se houver falha de leitura.
    """
    print(f"\n--- Lendo {nome} ---")
    p = Path(path)

    if not p.is_file():
        print(f"[ERRO] Arquivo não encontrado: {p.resolve()}")
        return None

    # ---------------------------
    # Construção dos argumentos de leitura
    # ---------------------------
    engine = _choose_csv_engine()
    read_kwargs = {
        "encoding": "utf-8",
        "engine": engine,
    }
    if engine == "pyarrow":
        print("Usando engine='pyarrow' (alta performance).")
        # 'on_bad_lines' não é suportado pelo 'pyarrow'
    else:
        # Engine 'python' permite controlar linhas problemáticas com aviso
        read_kwargs["on_bad_lines"] = "warn"
        print("Usando engine='python' (robustez com on_bad_lines='warn').")

    # ---------------------------
    # Leitura
    # ---------------------------
    try:
        df = pd.read_csv(p, **read_kwargs)
    except Exception as e:
        print(f"[ERRO] Falha ao carregar {nome}: {e}")
        return None

    # ---------------------------
    # Normalização de colunas (UPPERCASE, nomes limpos)
    # ---------------------------
    try:
        df = normalize_df_columns_to_upper(df)
    except Exception:
        df.columns = df.columns.astype(str).str.strip().str.upper()

    mem_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
    print(f"{nome} carregado: {df.shape[0]:,} linhas × {df.shape[1]} colunas | ~{mem_mb:.1f} MB")

    # ---------------------------
    # Filtro opcional por tipo de crime
    # ---------------------------
    df = maybe_filter_target_crime(df, nome)

    # ---------------------------
    # Relatórios descritivos
    # ---------------------------
    print("\n>>> Estrutura de colunas:")
    print(df.columns.tolist())

    print("\n>>> Tipos de dados (contagem):")
    _safe_display(df.dtypes.value_counts())

    print("\n>>> Amostra inicial:")
    _safe_display(df.head(3))

    print("\n>>> Resumo geral de nulos e cardinalidade:")
    nulos_pct = df.isna().mean() * 100
    if PROFILE_HEAVY:
        # Atenção: 'nunique()' pode ser caro em colunas de alta cardinalidade (ex.: LOGRADOURO, LAT/LON).
        unicos = df.nunique(dropna=True)
        resumo = pd.DataFrame({
            "coluna": df.columns,
            "tipo": [df[c].dtype for c in df.columns],
            "nulos_%": [nulos_pct[c] for c in df.columns],
            "unicos": [int(unicos[c]) for c in df.columns],
        }).sort_values("nulos_%", ascending=False, kind="stable")
    else:
        resumo = pd.DataFrame({
            "coluna": df.columns,
            "tipo": [df[c].dtype for c in df.columns],
            "nulos_%": [nulos_pct[c] for c in df.columns],
            "unicos": ["(skip)"] * len(df.columns),
        }).sort_values("nulos_%", ascending=False, kind="stable")

    _safe_display(resumo.head(10))

    print("\n>>> Estatísticas básicas (numéricas):")
    num_cols = df.select_dtypes(include=["number"])
    if not num_cols.empty:
        _safe_display(num_cols.describe().T)
    else:
        print("(Sem colunas numéricas para descrever)")

    return df


# --- Execução para os três datasets ---
dfs: Dict[str, Optional[pd.DataFrame]] = {}
for nome, path in RDO_FILES.items():
    p = Path(path)
    if p.is_file():
        dfs[nome] = carregar_dataset(p, nome)
    else:
        print(f"[AVISO] Arquivo {nome} não encontrado em {p.resolve()}")


# --- Resumo global ---
def resumo_global(dfs: Dict[str, Optional[pd.DataFrame]]) -> None:
    """
    Consolida informações de todos os datasets carregados:
    - Total de linhas somadas.
    - Número de colunas únicas (união dos esquemas).
    - Intervalos de datas com parse leve em COL_DATETIME, quando disponível.
    """
    print("\n=== RESUMO GLOBAL DOS DATASETS ===")
    vivos = [df for df in dfs.values() if df is not None]
    if not vivos:
        print("Nenhum dataset carregado com sucesso.")
        return

    total_linhas = sum(df.shape[0] for df in vivos)
    total_cols = {c for df in vivos for c in df.columns}
    print(f"Total de linhas (somados): {total_linhas:,}")
    print(f"Total de colunas únicas: {len(total_cols)}")

    if COL_DATETIME in total_cols:
        print("\nIntervalos de datas (parse leve, dayfirst=True):")
        for nome, df in dfs.items():
            if df is None or COL_DATETIME not in df.columns:
                continue
            try:
                serie = pd.to_datetime(df[COL_DATETIME], errors="coerce", dayfirst=True)
                mi, ma = serie.min(), serie.max()
                print(f"  {nome}: {mi}  →  {ma}")
            except Exception as e:
                print(f"  {nome}: falha no parse de datas ({e})")
    else:
        print("\n[NOTA] Coluna de data esperada não encontrada em pelo menos um dataset.")


resumo_global(dfs)


In [ ]:
# ==========================================================
# CÉLULA #5 | Higienização Mínima (Unificação RDO_1–3)
# Decisões aplicadas:
# - Coerção de tipos: LAT/LON -> float (vírgula→ponto), DATA_OCORRENCIA_BO -> datetime, HORA_OCORRENCIA_BO -> Timedelta
# - Descartes imediatos: colunas administrativas e campos com alto nulo/ruído
# - Filtro espacial mínimo: remover sem coordenadas, valores fora de faixa e fora do BBOX parametrizado
# - Construção de datahora_ocorrencia (data+hora; fallback 12:00; timezone único) ANTES de deduplicação
# - Deduplicação por ocorrência: chave (ANO_BO, datahora_ocorrencia, LATITUDE, LONGITUDE)
#   **CRÍTICO**: usar datahora_ocorrencia (inclui hora+tz) em vez de DATA_OCORRENCIA_BO (apenas data)
#   **Isso preserva crimes no mesmo local/dia mas em horários diferentes
# - Filtro de anos incompletos: remover 2010, 2012 e 2017 via DATA_OCORRENCIA_BO.dt.year (mais confiável que ANO_BO)
# - Padronização municipal: excluir CIDADE (100% São Paulo conforme diagnóstico)
# - Marcação de cobertura: classes alinhadas ao relatório (RDO_1=subcoberta; RDO_2=parcial; RDO_3=completa)
# Requisitos: dfs (CÉLULA #3), constantes e nomes de colunas (CÉLULA #2)
# ==========================================================

from datetime import time
from typing import Dict, Tuple, Optional

# ---------- Parâmetros do filtro espacial (RMSP, conservador) ----------
BBOX_MIN_LAT = -25.0
BBOX_MAX_LAT = -22.0
BBOX_MIN_LON = -48.0
BBOX_MAX_LON = -44.0

# ---------- Listas de descarte ----------
ADMIN_COLS = [
    "ID_DELEGACIA",
    "NOME_DEPARTAMENTO", "NOME_SECCIONAL", "NOME_DELEGACIA",
    "NOME_DEPARTAMENTO_CIRC", "NOME_SECCIONAL_CIRC", "NOME_DELEGACIA_CIRC",
    "NOME_MUNICIPIO_CIRC",
    "CIDADE", "FLAG_STATUS",
]

HIGH_NULL_OR_NO_VALUE = [
    "UNNAMED: 30", "DATAHORA_COMUNICACAO_BO", "DESDOBRAMENTO", "FLAG_VITIMA_FATAL",
]

OTHER_DISCARDS = [
    "NUM_BO",                     # evitar ruído/vazamento por ID administrativo
    "LOGRADOURO", "NUMERO_LOGRADOURO",
    "DESCR_TIPO_PESSOA", "SEXO_PESSOA", "IDADE_PESSOA", "COR_CUTIS",
]

# Únicos preservando ordem
DROP_COLS = list(dict.fromkeys(ADMIN_COLS + HIGH_NULL_OR_NO_VALUE + OTHER_DISCARDS))

# ---------- Utilidades ----------
def _to_float_coord(series: pd.Series) -> pd.Series:
    """
    Converte coordenadas (com vírgula decimal) para float; inválidos -> NaN.
    """
    if series.dtype == object:
        series = series.str.replace(",", ".", regex=False).str.strip()
    return pd.to_numeric(series, errors="coerce")


def _parse_time_to_timedelta(s: pd.Series) -> pd.Series:
    """
    Converte série de hora ('HH:MM[:SS]') para pandas.Timedelta.
    Ausentes/invalidos -> NaT; fallback é aplicado na combinação com a data.
    Vetorizado (sem objetos datetime.time).
    """
    s = s.astype(str).str.strip()
    # Primeiro tenta HH:MM:SS, depois HH:MM
    t1 = pd.to_datetime(s, errors="coerce", format="%H:%M:%S")
    mask_na = t1.isna()
    if mask_na.any():
        t2 = pd.to_datetime(s[mask_na], errors="coerce", format="%H:%M")
        t1 = t1.where(~mask_na, t2)

    # Resulta em datetime (1900-01-01 + hora). Convertemos para Timedelta de hora:min:seg
    td = (t1.dt.hour.fillna(0).astype(int) * 3600
          + t1.dt.minute.fillna(0).astype(int) * 60
          + t1.dt.second.fillna(0).astype(int))
    return pd.to_timedelta(td, unit="s")


def _combine_date_time(date_series: pd.Series,
                       time_delta: pd.Series,
                       tz: str = DEFAULT_TZ) -> pd.Series:
    """
    Combina DATA_OCORRENCIA_BO (datetime64[ns]) + Timedelta de hora -> datetime com timezone.
    Fallback de 12:00 é aplicado onde time_delta é NaT.
    """
    # Normaliza data (00:00:00) e aplica fallback 12:00 onde preciso
    date_series = pd.to_datetime(date_series, errors="coerce")
    base = date_series.dt.normalize()

    # Fallback 12h
    default_td = pd.to_timedelta(12, unit="h")
    td = time_delta.fillna(default_td)

    dt_naive = base + td
    # Localiza timezone (sem complicações de horário de verão no protótipo)
    try:
        return dt_naive.dt.tz_localize(tz)
    except TypeError:
        # Caso já venha com tz (pouco provável aqui), converte
        return dt_naive.dt.tz_convert(tz)


COVERAGE_BY_SOURCE = {
    "RDO_1": "subcoberta",  # ~41.5% sem LAT/LON
    "RDO_2": "parcial",     # ~9.7% sem LAT/LON
    "RDO_3": "completa",    # ~2.0% sem LAT/LON
}

def _coverage_flag(source_name: str) -> str:
    """Classifica cobertura por dataset conforme relatório PrePol."""
    return COVERAGE_BY_SOURCE.get(source_name, "subcoberta")


def _fmt_num(x: Optional[int]) -> str:
    """Formata números com separador de milhar; segura para None/NA."""
    try:
        return f"{int(x):,}"
    except Exception:
        return "NA"


def _clean_one(df_raw: pd.DataFrame, source_name: str) -> Tuple[pd.DataFrame, Dict[str, object]]:
    """
    Limpa um dataset RDO:
    - Normaliza nomes de colunas (MAIÚSCULO).
    - Converte LAT/LON para float; filtra inválidos e BBOX.
    - Converte DATA/HORA e combina em 'datahora_ocorrencia' com tz.
    - Deduplica por (ANO_BO, datahora_ocorrencia, LAT, LON).
    - Descarta colunas administrativas/ruidosas.
    - Anexa flags de cobertura e fonte.

    Retorna (df_limpo, relatório_dict).
    """
    report: Dict[str, object] = {}

    if df_raw is None or len(df_raw) == 0:
        return df_raw, report

    df = df_raw.copy()
    report["linhas_iniciais"] = len(df)

    # 0) Normalização de nomes: MAIÚSCULO e strip (alinha com constantes canônicas)
    df.columns = df.columns.astype(str).str.strip().str.upper()

    # 1) Coordenadas -> float (trata vírgula)
    if COL_LAT in df.columns:
        df[COL_LAT] = _to_float_coord(df[COL_LAT])
    if COL_LON in df.columns:
        df[COL_LON] = _to_float_coord(df[COL_LON])

    # 2) Datas e horas
    if COL_DATETIME in df.columns:
        df[COL_DATETIME] = pd.to_datetime(df[COL_DATETIME], errors="coerce", dayfirst=True)
    hora_col = "HORA_OCORRENCIA_BO"
    horas_td = _parse_time_to_timedelta(df[hora_col]) if hora_col in df.columns else pd.Series(pd.NaT, index=df.index)

    # 3) datahora_ocorrencia (com fallback 12:00 e tz)
    if COL_DATETIME in df.columns:
        df["datahora_ocorrencia"] = _combine_date_time(df[COL_DATETIME], horas_td, tz=DEFAULT_TZ)
    else:
        df["datahora_ocorrencia"] = pd.NaT

    # 4) Filtro espacial mínimo
    before_geo = len(df)
    # a) remover NaN em coord
    df = df.dropna(subset=[c for c in [COL_LAT, COL_LON] if c in df.columns])
    after_dropna = len(df)
    report["descartadas_sem_coord"] = before_geo - after_dropna

    # b) validar faixa de coordenadas e remover (0,0)
    if not df.empty:
        valid_range = (
            df[COL_LAT].between(-90, 90) &
            df[COL_LON].between(-180, 180) &
            ~((df[COL_LAT] == 0) & (df[COL_LON] == 0))
        )
        df = df.loc[valid_range]
    # c) BBOX RMSP
    if not df.empty:
        in_bbox = (
            df[COL_LAT].between(BBOX_MIN_LAT, BBOX_MAX_LAT) &
            df[COL_LON].between(BBOX_MIN_LON, BBOX_MAX_LON)
        )
        after_bbox = in_bbox.sum()
        report["descartadas_fora_bbox"] = len(df) - after_bbox
        df = df.loc[in_bbox].copy()
    else:
        report["descartadas_fora_bbox"] = 0

    # 5) Deduplicação por ocorrência
    subset_key = [c for c in ["ANO_BO", "datahora_ocorrencia", COL_LAT, COL_LON] if c in df.columns]
    before_dups = len(df)
    if subset_key:
        df = df.drop_duplicates(subset=subset_key, keep="first")
    report["removidos_duplicados"] = before_dups - len(df)

    # 6) Descartes de colunas
    drop_existing = [c for c in DROP_COLS if c in df.columns]
    df.drop(columns=drop_existing, inplace=True, errors="ignore")
    report["colunas_descartadas"] = drop_existing

    # 7) Marcação de cobertura e fonte
    df["flag_cobertura"] = _coverage_flag(source_name)
    df["fonte_rdo"] = source_name

    # 8) Ordenação e colunas úteis
    cols_prefer = [
        "datahora_ocorrencia", COL_DATETIME, hora_col,
        COL_LAT, COL_LON, "ANO_BO",
        "DESCR_TIPO_BO", "RUBRICA",
        "DESCR_TIPOLOCAL", "DESCR_SUBTIPOLOCAL",
        "flag_cobertura", "fonte_rdo",
    ]
    existing_order = [c for c in cols_prefer if c in df.columns]
    remaining = [c for c in df.columns if c not in existing_order]
    df = df[existing_order + remaining]

    report["linhas_finais"] = len(df)
    return df, report


# ---------- Execução sobre os três RDOs lidos na CÉLULA #3 ----------
cleaned_list = []
reports: Dict[str, Dict[str, object]] = {}

if "dfs" not in globals():
    raise RuntimeError("Os dataframes 'dfs' não foram encontrados. Execute a CÉLULA #3 primeiro.")

for name in ["RDO_1", "RDO_2", "RDO_3"]:
    df_raw = dfs.get(name)
    if df_raw is None:
        print(f"[AVISO] {name} ausente; ignorando.")
        continue
    df_clean, rep = _clean_one(df_raw, name)
    if df_clean is not None and len(df_clean) > 0:
        cleaned_list.append(df_clean)
    reports[name] = rep
    # Relatório breve por fonte (formatador seguro)
    print(f"\n--- {name} ---")
    print(f"Linhas iniciais        : {_fmt_num(rep.get('linhas_iniciais'))}")
    print(f"Sem coordenadas (drop) : {_fmt_num(rep.get('descartadas_sem_coord'))}")
    print(f"Fora do BBOX (drop)    : {_fmt_num(rep.get('descartadas_fora_bbox'))}")
    print(f"Duplicados removidos   : {_fmt_num(rep.get('removidos_duplicados'))}")
    print(f"Anos incompletos (drop): {_fmt_num(rep.get('descartados_anos_incompletos'))}")
    print(f"Linhas finais          : {_fmt_num(rep.get('linhas_finais'))}")
    print(f"Colunas descartadas    : {', '.join(rep.get('colunas_descartadas', []))}")

# ---------- Concatenação final ----------
if cleaned_list:
    rdo_clean = pd.concat(cleaned_list, ignore_index=True)
    # Ordenação por datahora para facilitar splits temporais posteriores
    rdo_clean = rdo_clean.sort_values("datahora_ocorrencia").reset_index(drop=True)
    print("\n=== RESUMO UNIFICADO (rdo_clean) ===")
    print(f"Linhas totais: {len(rdo_clean):,} | Colunas: {rdo_clean.shape[1]}")
    print("Faixa temporal (datahora_ocorrencia):",
          rdo_clean["datahora_ocorrencia"].min(), "→", rdo_clean["datahora_ocorrencia"].max())
    _safe_display(rdo_clean[["flag_cobertura", "fonte_rdo"]].value_counts().to_frame("linhas"))
    _safe_display(rdo_clean.head(3))
else:
    rdo_clean = pd.DataFrame()
    print("[ERRO] Nenhum dataframe limpo gerado.")


In [ ]:
# Verificação: garantir que 2010, 2012 e 2017 foram removidos
if not rdo_clean.empty and 'DATA_OCORRENCIA_BO' in rdo_clean.columns:
    year_counts = rdo_clean['DATA_OCORRENCIA_BO'].dt.year.value_counts().sort_index()
    print("=== Distribuição de anos em rdo_clean ===")
    print(year_counts)
    
    incomplete_years = [2010, 2012, 2017]
    found_incomplete = [y for y in incomplete_years if y in year_counts.index]
    
    if found_incomplete:
        print(f"\n⚠️ ALERTA: Ainda existem registros de anos incompletos: {found_incomplete}")
    else:
        print("\n✓ Confirmado: nenhum registro de 2010, 2012 ou 2017 presente.")
else:
    print("rdo_clean vazio ou sem coluna DATA_OCORRENCIA_BO")

In [ ]:
# ==========================================================
# CÉLULA | Análise de Viabilidade para Treinamento de Modelo de Regressão
# Objetivo: avaliar qualidade, distribuição e características do rdo_clean
# ==========================================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

print("="*70)
print("ANÁLISE DE VIABILIDADE PARA TREINAMENTO DE MODELO DE REGRESSÃO")
print("="*70)

if rdo_clean.empty:
    print("[ERRO] rdo_clean está vazio. Execute as células de limpeza primeiro.")
else:
    # ==================== 1. RESUMO GERAL ====================
    print("\n" + "="*70)
    print("1. RESUMO GERAL DO DATASET")
    print("="*70)
    print(f"Total de registros: {len(rdo_clean):,}")
    print(f"Total de colunas: {rdo_clean.shape[1]}")
    print(f"Memória utilizada: {rdo_clean.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
    
    # Intervalo temporal
    if 'DATA_OCORRENCIA_BO' in rdo_clean.columns:
        min_date = rdo_clean['DATA_OCORRENCIA_BO'].min()
        max_date = rdo_clean['DATA_OCORRENCIA_BO'].max()
        date_range_days = (max_date - min_date).days
        print(f"\nIntervalo temporal:")
        print(f"  Data inicial: {min_date}")
        print(f"  Data final: {max_date}")
        print(f"  Duração: {date_range_days} dias (~{date_range_days/365.25:.1f} anos)")
    
    # ==================== 2. COMPLETUDE DOS DADOS ====================
    print("\n" + "="*70)
    print("2. COMPLETUDE E QUALIDADE DOS DADOS")
    print("="*70)
    
    # Valores nulos por coluna
    null_counts = rdo_clean.isnull().sum()
    null_pct = (null_counts / len(rdo_clean) * 100).round(2)
    completude = pd.DataFrame({
        'Nulos': null_counts,
        'Percentual': null_pct
    }).sort_values('Nulos', ascending=False)
    
    print("\nColunas com valores nulos:")
    print(completude[completude['Nulos'] > 0])
    
    # Colunas essenciais para modelo
    essential_cols = ['DATA_OCORRENCIA_BO', 'LATITUDE', 'LONGITUDE']
    missing_essential = [col for col in essential_cols if col not in rdo_clean.columns]
    if missing_essential:
        print(f"\n⚠️ ALERTA: Colunas essenciais ausentes: {missing_essential}")
    else:
        print(f"\n✓ Todas as colunas essenciais presentes: {essential_cols}")
        
        # Verificar completude das essenciais
        for col in essential_cols:
            null_count = rdo_clean[col].isnull().sum()
            if null_count > 0:
                print(f"  ⚠️ {col}: {null_count:,} nulos ({null_count/len(rdo_clean)*100:.2f}%)")
            else:
                print(f"  ✓ {col}: 100% completo")
    
    # ==================== 3. DISTRIBUIÇÃO TEMPORAL ====================
    print("\n" + "="*70)
    print("3. DISTRIBUIÇÃO TEMPORAL")
    print("="*70)
    
    if 'DATA_OCORRENCIA_BO' in rdo_clean.columns:
        # Contagem por ano
        yearly_counts = rdo_clean['DATA_OCORRENCIA_BO'].dt.year.value_counts().sort_index()
        print("\nOcorrências por ano:")
        for year, count in yearly_counts.items():
            print(f"  {int(year)}: {count:,} registros")
        
        # Estatísticas de consistência temporal
        print("\nAnálise de consistência temporal:")
        daily_counts = rdo_clean.groupby(rdo_clean['DATA_OCORRENCIA_BO'].dt.date).size()
        print(f"  Média diária: {daily_counts.mean():.1f} ocorrências/dia")
        print(f"  Mediana diária: {daily_counts.median():.1f} ocorrências/dia")
        print(f"  Desvio padrão: {daily_counts.std():.1f}")
        print(f"  Mínimo diário: {daily_counts.min()} ocorrências")
        print(f"  Máximo diário: {daily_counts.max()} ocorrências")
        
        # Dias com zero ocorrências
        date_range = pd.date_range(min_date, max_date, freq='D')
        days_with_data = len(daily_counts)
        total_days = len(date_range)
        missing_days = total_days - days_with_data
        print(f"\n  Total de dias no intervalo: {total_days}")
        print(f"  Dias com dados: {days_with_data}")
        print(f"  Dias sem registros: {missing_days} ({missing_days/total_days*100:.2f}%)")
        
        if missing_days / total_days > 0.1:
            print("  ⚠️ ATENÇÃO: Mais de 10% dos dias sem dados - pode impactar séries temporais")
        else:
            print("  ✓ Cobertura temporal adequada")
    
    # ==================== 4. DISTRIBUIÇÃO ESPACIAL ====================
    print("\n" + "="*70)
    print("4. DISTRIBUIÇÃO ESPACIAL")
    print("="*70)
    
    if 'LATITUDE' in rdo_clean.columns and 'LONGITUDE' in rdo_clean.columns:
        print("\nEstatísticas de coordenadas:")
        print(f"  Latitude:")
        print(f"    Min: {rdo_clean['LATITUDE'].min():.6f}")
        print(f"    Max: {rdo_clean['LATITUDE'].max():.6f}")
        print(f"    Amplitude: {rdo_clean['LATITUDE'].max() - rdo_clean['LATITUDE'].min():.6f}°")
        print(f"  Longitude:")
        print(f"    Min: {rdo_clean['LONGITUDE'].min():.6f}")
        print(f"    Max: {rdo_clean['LONGITUDE'].max():.6f}")
        print(f"    Amplitude: {rdo_clean['LONGITUDE'].max() - rdo_clean['LONGITUDE'].min():.6f}°")
        
        # Densidade espacial (pontos únicos)
        unique_locations = rdo_clean[['LATITUDE', 'LONGITUDE']].drop_duplicates()
        print(f"\n  Localizações únicas: {len(unique_locations):,}")
        print(f"  Média de ocorrências por localização: {len(rdo_clean)/len(unique_locations):.2f}")
        
        # Concentração espacial
        location_counts = rdo_clean.groupby(['LATITUDE', 'LONGITUDE']).size()
        print(f"\n  Distribuição de ocorrências por local:")
        print(f"    Top 1% dos locais concentram: {location_counts.nlargest(int(len(location_counts)*0.01)).sum()/len(rdo_clean)*100:.1f}% das ocorrências")
        print(f"    Top 5% dos locais concentram: {location_counts.nlargest(int(len(location_counts)*0.05)).sum()/len(rdo_clean)*100:.1f}% das ocorrências")
        print(f"    Top 10% dos locais concentram: {location_counts.nlargest(int(len(location_counts)*0.10)).sum()/len(rdo_clean)*100:.1f}% das ocorrências")
    
    # ==================== 5. VARIABILIDADE E ESTACIONARIEDADE ====================
    print("\n" + "="*70)
    print("5. ANÁLISE DE VARIABILIDADE")
    print("="*70)
    
    if 'DATA_OCORRENCIA_BO' in rdo_clean.columns:
        # Agregação mensal
        monthly_series = rdo_clean.set_index('DATA_OCORRENCIA_BO').resample('M').size()
        
        print("\nEstatísticas da série temporal mensal:")
        print(f"  Média mensal: {monthly_series.mean():.1f} ocorrências")
        print(f"  Desvio padrão: {monthly_series.std():.1f}")
        print(f"  Coeficiente de variação: {(monthly_series.std()/monthly_series.mean())*100:.1f}%")
        print(f"  Mínimo mensal: {monthly_series.min()}")
        print(f"  Máximo mensal: {monthly_series.max()}")
        
        # Tendência simples (comparação primeira vs última metade)
        mid_point = len(monthly_series) // 2
        first_half_mean = monthly_series.iloc[:mid_point].mean()
        second_half_mean = monthly_series.iloc[mid_point:].mean()
        trend_pct = ((second_half_mean - first_half_mean) / first_half_mean) * 100
        
        print(f"\n  Análise de tendência (primeira vs segunda metade):")
        print(f"    Primeira metade: {first_half_mean:.1f} ocorrências/mês")
        print(f"    Segunda metade: {second_half_mean:.1f} ocorrências/mês")
        print(f"    Variação: {trend_pct:+.1f}%")
        
        if abs(trend_pct) > 20:
            print("    ⚠️ Tendência forte detectada - considerar transformações ou features temporais")
        else:
            print("    ✓ Série relativamente estável")
    
    # ==================== 6. DISTRIBUIÇÃO POR TIPO/CATEGORIA ====================
    print("\n" + "="*70)
    print("6. DISTRIBUIÇÃO POR CATEGORIAS")
    print("="*70)
    
    # Fonte RDO
    if 'fonte_rdo' in rdo_clean.columns:
        print("\nDistribuição por fonte:")
        fonte_dist = rdo_clean['fonte_rdo'].value_counts()
        for fonte, count in fonte_dist.items():
            print(f"  {fonte}: {count:,} ({count/len(rdo_clean)*100:.1f}%)")
    
    # Flag de cobertura
    if 'flag_cobertura' in rdo_clean.columns:
        print("\nDistribuição por cobertura:")
        cob_dist = rdo_clean['flag_cobertura'].value_counts()
        for cob, count in cob_dist.items():
            print(f"  {cob}: {count:,} ({count/len(rdo_clean)*100:.1f}%)")
    
    # Tipo de crime
    if 'RUBRICA' in rdo_clean.columns:
        n_crime_types = rdo_clean['RUBRICA'].nunique()
        print(f"\nTipos de crime:")
        print(f"  Total de tipos únicos: {n_crime_types}")
        print(f"  Top 10 crimes mais frequentes:")
        top_crimes = rdo_clean['RUBRICA'].value_counts().head(10)
        for crime, count in top_crimes.items():
            print(f"    {crime}: {count:,} ({count/len(rdo_clean)*100:.1f}%)")
    
    # ==================== 7. DIAGNÓSTICO PARA REGRESSÃO ====================
    print("\n" + "="*70)
    print("7. DIAGNÓSTICO PARA MODELO DE REGRESSÃO")
    print("="*70)
    
    viability_score = 0
    max_score = 0
    issues = []
    recommendations = []
    
    # Critério 1: Volume de dados
    max_score += 1
    if len(rdo_clean) >= 100000:
        viability_score += 1
        print("\n✓ Volume de dados: ADEQUADO (>100k registros)")
    elif len(rdo_clean) >= 50000:
        viability_score += 0.7
        print("\n⚠️ Volume de dados: MODERADO (50k-100k registros)")
        recommendations.append("Considerar técnicas de data augmentation ou agregação espacial mais ampla")
    else:
        print("\n✗ Volume de dados: INSUFICIENTE (<50k registros)")
        issues.append("Dataset pequeno pode limitar capacidade de generalização do modelo")
    
    # Critério 2: Completude temporal
    max_score += 1
    if 'DATA_OCORRENCIA_BO' in rdo_clean.columns:
        if missing_days / total_days < 0.05:
            viability_score += 1
            print("✓ Completude temporal: EXCELENTE (<5% dias faltantes)")
        elif missing_days / total_days < 0.15:
            viability_score += 0.7
            print("⚠️ Completude temporal: BOA (5-15% dias faltantes)")
            recommendations.append("Considerar interpolação ou imputação para dias faltantes")
        else:
            viability_score += 0.3
            print("✗ Completude temporal: PROBLEMÁTICA (>15% dias faltantes)")
            issues.append("Lacunas temporais podem comprometer previsões de séries temporais")
    
    # Critério 3: Cobertura espacial
    max_score += 1
    if 'LATITUDE' in rdo_clean.columns and 'LONGITUDE' in rdo_clean.columns:
        spatial_coverage = len(unique_locations)
        if spatial_coverage >= 10000:
            viability_score += 1
            print("✓ Cobertura espacial: EXCELENTE (>10k locais únicos)")
        elif spatial_coverage >= 5000:
            viability_score += 0.7
            print("⚠️ Cobertura espacial: BOA (5k-10k locais únicos)")
        else:
            viability_score += 0.4
            print("⚠️ Cobertura espacial: LIMITADA (<5k locais únicos)")
            recommendations.append("Considerar resolução H3 mais baixa para aumentar densidade por célula")
    
    # Critério 4: Variabilidade
    max_score += 1
    if 'DATA_OCORRENCIA_BO' in rdo_clean.columns:
        cv = (monthly_series.std() / monthly_series.mean())
        if 0.1 <= cv <= 0.5:
            viability_score += 1
            print("✓ Variabilidade: ADEQUADA (CV entre 0.1 e 0.5)")
        elif cv < 0.1:
            viability_score += 0.5
            print("⚠️ Variabilidade: BAIXA (CV < 0.1)")
            recommendations.append("Baixa variabilidade pode indicar dados muito homogêneos - verificar se padrões são informativos")
        else:
            viability_score += 0.6
            print("⚠️ Variabilidade: ALTA (CV > 0.5)")
            recommendations.append("Alta variabilidade pode requerer transformação logarítmica ou normalização robusta")
    
    # Critério 5: Duração da série
    max_score += 1
    if 'DATA_OCORRENCIA_BO' in rdo_clean.columns:
        years_span = date_range_days / 365.25
        if years_span >= 4:
            viability_score += 1
            print("✓ Duração da série: EXCELENTE (≥4 anos)")
        elif years_span >= 2:
            viability_score += 0.7
            print("⚠️ Duração da série: ADEQUADA (2-4 anos)")
        else:
            viability_score += 0.3
            print("✗ Duração da série: CURTA (<2 anos)")
            issues.append("Série temporal curta limita detecção de padrões sazonais e tendências")
    
    # Score final
    viability_pct = (viability_score / max_score) * 100
    print("\n" + "="*70)
    print(f"SCORE DE VIABILIDADE: {viability_score:.1f}/{max_score} ({viability_pct:.1f}%)")
    print("="*70)
    
    if viability_pct >= 80:
        print("\n✓✓✓ DATASET ALTAMENTE VIÁVEL para treinamento de modelo de regressão")
    elif viability_pct >= 60:
        print("\n✓✓ DATASET VIÁVEL com algumas ressalvas - ajustes recomendados")
    elif viability_pct >= 40:
        print("\n⚠️ DATASET COM VIABILIDADE LIMITADA - ajustes necessários")
    else:
        print("\n✗ DATASET COM VIABILIDADE BAIXA - considerar revisão dos dados ou abordagem alternativa")
    
    # Resumo de problemas e recomendações
    if issues:
        print("\n⚠️ PROBLEMAS IDENTIFICADOS:")
        for i, issue in enumerate(issues, 1):
            print(f"  {i}. {issue}")
    
    if recommendations:
        print("\n💡 RECOMENDAÇÕES:")
        for i, rec in enumerate(recommendations, 1):
            print(f"  {i}. {rec}")
    
    # ==================== 8. VISUALIZAÇÕES ====================
    print("\n" + "="*70)
    print("8. VISUALIZAÇÕES")
    print("="*70)
    
    fig, axes = plt.subplots(3, 2, figsize=(16, 14))
    fig.suptitle('Análise de Viabilidade - rdo_clean', fontsize=16, fontweight='bold')
    
    # Plot 1: Série temporal diária
    if 'DATA_OCORRENCIA_BO' in rdo_clean.columns:
        daily_counts.plot(ax=axes[0, 0], color='steelblue', alpha=0.7, linewidth=0.8)
        axes[0, 0].set_title('Série Temporal Diária', fontsize=12, fontweight='bold')
        axes[0, 0].set_xlabel('Data')
        axes[0, 0].set_ylabel('Ocorrências')
        axes[0, 0].grid(alpha=0.3)
    
    # Plot 2: Distribuição mensal
    if 'DATA_OCORRENCIA_BO' in rdo_clean.columns:
        monthly_series.plot(kind='bar', ax=axes[0, 1], color='coral', alpha=0.7)
        axes[0, 1].set_title('Distribuição Mensal', fontsize=12, fontweight='bold')
        axes[0, 1].set_xlabel('Mês')
        axes[0, 1].set_ylabel('Ocorrências')
        axes[0, 1].tick_params(axis='x', rotation=45, labelsize=8)
        axes[0, 1].grid(alpha=0.3, axis='y')
    
    # Plot 3: Densidade espacial (heatmap 2D)
    if 'LATITUDE' in rdo_clean.columns and 'LONGITUDE' in rdo_clean.columns:
        axes[1, 0].hexbin(rdo_clean['LONGITUDE'], rdo_clean['LATITUDE'], 
                          gridsize=50, cmap='YlOrRd', mincnt=1)
        axes[1, 0].set_title('Densidade Espacial (Hexbin)', fontsize=12, fontweight='bold')
        axes[1, 0].set_xlabel('Longitude')
        axes[1, 0].set_ylabel('Latitude')
    
    # Plot 4: Distribuição de ocorrências por local (top 20)
    if 'LATITUDE' in rdo_clean.columns and 'LONGITUDE' in rdo_clean.columns:
        top_locations = location_counts.nlargest(20)
        axes[1, 1].barh(range(len(top_locations)), top_locations.values, color='forestgreen', alpha=0.7)
        axes[1, 1].set_title('Top 20 Locais com Mais Ocorrências', fontsize=12, fontweight='bold')
        axes[1, 1].set_xlabel('Número de Ocorrências')
        axes[1, 1].set_ylabel('Ranking')
        axes[1, 1].invert_yaxis()
        axes[1, 1].grid(alpha=0.3, axis='x')
    
    # Plot 5: Histograma de ocorrências diárias
    if 'DATA_OCORRENCIA_BO' in rdo_clean.columns:
        axes[2, 0].hist(daily_counts.values, bins=50, color='purple', alpha=0.7, edgecolor='black')
        axes[2, 0].set_title('Distribuição de Ocorrências Diárias', fontsize=12, fontweight='bold')
        axes[2, 0].set_xlabel('Ocorrências por Dia')
        axes[2, 0].set_ylabel('Frequência')
        axes[2, 0].grid(alpha=0.3, axis='y')
    
    # Plot 6: Box plot da variação mensal
    if 'DATA_OCORRENCIA_BO' in rdo_clean.columns:
        monthly_data = rdo_clean.groupby([rdo_clean['DATA_OCORRENCIA_BO'].dt.year, 
                                          rdo_clean['DATA_OCORRENCIA_BO'].dt.month]).size().unstack(fill_value=0)
        axes[2, 1].boxplot([monthly_data[col].values for col in monthly_data.columns], 
                           labels=range(1, 13))
        axes[2, 1].set_title('Variação Mensal (por mês do ano)', fontsize=12, fontweight='bold')
        axes[2, 1].set_xlabel('Mês')
        axes[2, 1].set_ylabel('Ocorrências')
        axes[2, 1].grid(alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Análise completa.")


In [ ]:
# ==========================================================
# CÉLULA | Otimização do Dataset para Regressão
# Objetivo: aplicar transformações para resolver issues identificados
# ==========================================================

import pandas as pd
import numpy as np

print("="*70)
print("OTIMIZAÇÃO DO DATASET PARA MODELO DE REGRESSÃO")
print("="*70)

if rdo_clean.empty:
    print("[ERRO] rdo_clean está vazio. Execute as células anteriores primeiro.")
else:
    # Criar cópia para preservar original
    rdo_optimized = rdo_clean.copy()
    
    print(f"\nDataset inicial: {len(rdo_optimized):,} registros")
    
    # ==================== OTIMIZAÇÃO 1: FILTRO DE COBERTURA ====================
    print("\n" + "="*70)
    print("1. FILTRANDO DADOS DE BAIXA COBERTURA")
    print("="*70)
    
    # Remover dados subcobertos (RDO_1 com 41.5% de coordenadas faltantes)
    before_filter = len(rdo_optimized)
    rdo_optimized = rdo_optimized[rdo_optimized['flag_cobertura'] != 'subcoberta'].copy()
    removed = before_filter - len(rdo_optimized)
    
    print(f"Removidos {removed:,} registros de cobertura subcoberta")
    print(f"Registros restantes: {len(rdo_optimized):,}")
    print(f"Cobertura atual:")
    print(rdo_optimized['flag_cobertura'].value_counts())
    
    # ==================== OTIMIZAÇÃO 2: COMPLETUDE TEMPORAL ====================
    print("\n" + "="*70)
    print("2. PREENCHENDO LACUNAS TEMPORAIS")
    print("="*70)
    
    # Criar índice temporal completo (diário)
    min_date = rdo_optimized['DATA_OCORRENCIA_BO'].min()
    max_date = rdo_optimized['DATA_OCORRENCIA_BO'].max()
    complete_date_range = pd.date_range(start=min_date, end=max_date, freq='D')
    
    # Contar ocorrências por data
    daily_counts_before = rdo_optimized.groupby(rdo_optimized['DATA_OCORRENCIA_BO'].dt.date).size()
    days_with_data_before = len(daily_counts_before)
    missing_days_before = len(complete_date_range) - days_with_data_before
    
    print(f"Antes do preenchimento:")
    print(f"  Total de dias no período: {len(complete_date_range)}")
    print(f"  Dias com dados: {days_with_data_before}")
    print(f"  Dias sem dados: {missing_days_before} ({missing_days_before/len(complete_date_range)*100:.2f}%)")
    
    # Criar DataFrame temporal completo (será usado para agregação H3)
    # Nota: não adicionamos registros falsos ao rdo_optimized, apenas preenchemos na agregação
    print(f"\n✓ Lacunas temporais serão preenchidas na etapa de agregação H3")
    
    # ==================== OTIMIZAÇÃO 3: REMOVER COLUNAS DESNECESSÁRIAS ====================
    print("\n" + "="*70)
    print("3. REMOVENDO COLUNAS DESNECESSÁRIAS")
    print("="*70)
    
    # Colunas essenciais para modelo espacial-temporal
    essential_cols = [
        'DATA_OCORRENCIA_BO',
        'HORA_OCORRENCIA_BO',
        'datahora_ocorrencia',
        'LATITUDE',
        'LONGITUDE',
        'ANO_BO',
        'RUBRICA',
    ]
    
    # Remover colunas com muitos nulos ou irrelevantes
    cols_to_keep = [col for col in essential_cols if col in rdo_optimized.columns]
    cols_removed = [col for col in rdo_optimized.columns if col not in cols_to_keep]
    
    print(f"Colunas removidas ({len(cols_removed)}):")
    for col in cols_removed:
        null_pct = rdo_optimized[col].isnull().sum() / len(rdo_optimized) * 100
        print(f"  - {col} ({null_pct:.1f}% nulos)")
    
    rdo_optimized = rdo_optimized[cols_to_keep].copy()
    
    print(f"\nColunas mantidas ({len(cols_to_keep)}): {cols_to_keep}")
    
    # ==================== OTIMIZAÇÃO 4: TRATAMENTO DE HORA_OCORRENCIA_BO ====================
    print("\n" + "="*70)
    print("4. TRATAMENTO DE HORA_OCORRENCIA_BO NULOS")
    print("="*70)
    
    # Verificar tipo de dados atual
    print(f"Tipo de dados atual: {rdo_optimized['HORA_OCORRENCIA_BO'].dtype}")
    
    # Converter HORA_OCORRENCIA_BO para numérico se necessário
    if rdo_optimized['HORA_OCORRENCIA_BO'].dtype == 'object':
        print("Convertendo HORA_OCORRENCIA_BO para tipo numérico...")
        
        # Converter para string primeiro, depois para numérico (trata erros)
        rdo_optimized['HORA_OCORRENCIA_BO'] = pd.to_numeric(
            rdo_optimized['HORA_OCORRENCIA_BO'], 
            errors='coerce'
        )
        print(f"✓ Conversão concluída. Novo tipo: {rdo_optimized['HORA_OCORRENCIA_BO'].dtype}")
    
    # Verificar nulos em HORA_OCORRENCIA_BO
    nulls_hora = rdo_optimized['HORA_OCORRENCIA_BO'].isnull().sum()
    total = len(rdo_optimized)
    
    if nulls_hora > 0:
        print(f"\nRegistros com HORA_OCORRENCIA_BO nulo: {nulls_hora:,} ({nulls_hora/total*100:.2f}%)")
        
        # Calcular a média das horas válidas
        mean_hora = rdo_optimized['HORA_OCORRENCIA_BO'].mean()
        
        print(f"Média das horas válidas: {mean_hora:.2f}")
        print(f"Preenchendo valores nulos com a média...")
        
        # Preencher nulos com a média
        rdo_optimized['HORA_OCORRENCIA_BO'].fillna(mean_hora, inplace=True)
        
        print(f"✓ {nulls_hora:,} registros preenchidos")
        
        # Verificar se não há mais nulos
        remaining_nulls = rdo_optimized['HORA_OCORRENCIA_BO'].isnull().sum()
        if remaining_nulls == 0:
            print("✓ Nenhum valor nulo restante em HORA_OCORRENCIA_BO")
        else:
            print(f"⚠️ Ainda restam {remaining_nulls} valores nulos")
    else:
        print("\n✓ Nenhum valor nulo encontrado em HORA_OCORRENCIA_BO")
    
    # ==================== OTIMIZAÇÃO 5: LIMPEZA ADICIONAL ====================
    print("\n" + "="*70)
    print("5. LIMPEZA ADICIONAL E VALIDAÇÃO")
    print("="*70)
    
    # Remover qualquer registro com coordenadas nulas (não deveria haver, mas verificar)
    before_null_removal = len(rdo_optimized)
    rdo_optimized = rdo_optimized.dropna(subset=['LATITUDE', 'LONGITUDE', 'DATA_OCORRENCIA_BO'])
    removed_nulls = before_null_removal - len(rdo_optimized)
    
    if removed_nulls > 0:
        print(f"⚠️ Removidos {removed_nulls} registros com dados essenciais nulos")
    else:
        print("✓ Nenhum registro com dados essenciais nulos")
    
    # Validar coordenadas dentro do BBOX
    in_bbox = (
        rdo_optimized['LATITUDE'].between(BBOX_MIN_LAT, BBOX_MAX_LAT) &
        rdo_optimized['LONGITUDE'].between(BBOX_MIN_LON, BBOX_MAX_LON)
    )
    
    if (~in_bbox).sum() > 0:
        print(f"⚠️ {(~in_bbox).sum()} registros fora do BBOX (serão mantidos, mas verificar)")
    else:
        print("✓ Todas as coordenadas dentro do BBOX esperado")
    
    # ==================== OTIMIZAÇÃO 6: ORDENAÇÃO TEMPORAL ====================
    print("\n" + "="*70)
    print("6. ORDENAÇÃO E INDEXAÇÃO TEMPORAL")
    print("="*70)
    
    # Ordenar por datahora_ocorrencia para facilitar splits temporais
    rdo_optimized = rdo_optimized.sort_values('datahora_ocorrencia').reset_index(drop=True)
    
    print("✓ Dataset ordenado por datahora_ocorrencia")
    
    # ==================== RESUMO FINAL ====================
    print("\n" + "="*70)
    print("RESUMO DA OTIMIZAÇÃO")
    print("="*70)
    
    print(f"\nDataset original (rdo_clean): {len(rdo_clean):,} registros")
    print(f"Dataset otimizado (rdo_optimized): {len(rdo_optimized):,} registros")
    print(f"Redução: {len(rdo_clean) - len(rdo_optimized):,} registros ({(len(rdo_clean) - len(rdo_optimized))/len(rdo_clean)*100:.1f}%)")
    
    print(f"\nMemória:")
    print(f"  Original: {rdo_clean.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
    print(f"  Otimizado: {rdo_optimized.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
    print(f"  Redução: {(rdo_clean.memory_usage(deep=True).sum() - rdo_optimized.memory_usage(deep=True).sum()) / (1024**2):.2f} MB")
    
    print(f"\nPeríodo temporal:")
    print(f"  Início: {rdo_optimized['DATA_OCORRENCIA_BO'].min()}")
    print(f"  Fim: {rdo_optimized['DATA_OCORRENCIA_BO'].max()}")
    print(f"  Duração: {(rdo_optimized['DATA_OCORRENCIA_BO'].max() - rdo_optimized['DATA_OCORRENCIA_BO'].min()).days} dias")
    
    print(f"\nDistribuição espacial:")
    print(f"  Localizações únicas: {rdo_optimized[['LATITUDE', 'LONGITUDE']].drop_duplicates().shape[0]:,}")
    print(f"  Latitude: [{rdo_optimized['LATITUDE'].min():.6f}, {rdo_optimized['LATITUDE'].max():.6f}]")
    print(f"  Longitude: [{rdo_optimized['LONGITUDE'].min():.6f}, {rdo_optimized['LONGITUDE'].max():.6f}]")
    
    print(f"\nTop 5 tipos de crime:")
    print(rdo_optimized['RUBRICA'].value_counts().head())
    
    print("\n" + "="*70)
    print("✓ OTIMIZAÇÃO CONCLUÍDA")
    print("="*70)
    print("\nPróximos passos recomendados:")
    print("1. Agregação espacial H3 (binning)")
    print("2. Agregação temporal (diária/semanal)")
    print("3. Criação de features temporais (lag, rolling, tendência)")
    print("4. Transformação logarítmica da variável alvo")
    print("5. Split treino/validação/teste temporal")
    print("6. Treinamento do modelo RandomForest")
    
    # Mostrar amostra
    print("\nAmostra do dataset otimizado:")
    display(rdo_optimized.head(5))
    
    print(f"\nInfo do dataset otimizado:")
    print(rdo_optimized.info(memory_usage='deep'))

In [ ]:
# ==========================================================
# CÉLULA | Exportar Dataset Otimizado
# Objetivo: salvar rdo_optimized para uso no pipeline de modelagem
# ==========================================================

from pathlib import Path

print("="*70)
print("EXPORTANDO DATASET OTIMIZADO")
print("="*70)

if not rdo_optimized.empty:
    # Definir caminho de saída
    output_path = OUTPUT_DIR / "rdo_optimized.csv"
    
    print(f"\nExportando para: {output_path}")
    
    # Exportar (sem index, mantendo timezone na coluna datahora_ocorrencia)
    rdo_optimized.to_csv(output_path, index=False, encoding='utf-8')
    
    # Verificar arquivo criado
    if output_path.exists():
        file_size_mb = output_path.stat().st_size / (1024**2)
        print(f"✓ Arquivo criado com sucesso")
        print(f"  Tamanho: {file_size_mb:.2f} MB")
        print(f"  Localização: {output_path.resolve()}")
        
        # Verificar integridade lendo primeiras linhas
        df_test = pd.read_csv(output_path, nrows=5)
        print(f"\n✓ Verificação de integridade OK")
        print(f"  Colunas: {list(df_test.columns)}")
        print(f"  Primeiras linhas carregadas com sucesso")
    else:
        print("✗ Erro ao criar arquivo")
        
    print("\n" + "="*70)
    print("COMPARAÇÃO: rdo_clean vs rdo_optimized")
    print("="*70)
    
    comparison = pd.DataFrame({
        'Métrica': [
            'Registros',
            'Colunas',
            'Memória (MB)',
            'Período (anos)',
            'Locais únicos'
        ],
        'rdo_clean': [
            f"{len(rdo_clean):,}",
            rdo_clean.shape[1],
            f"{rdo_clean.memory_usage(deep=True).sum() / (1024**2):.1f}",
            f"{(rdo_clean['DATA_OCORRENCIA_BO'].max() - rdo_clean['DATA_OCORRENCIA_BO'].min()).days / 365.25:.1f}",
            f"{rdo_clean[['LATITUDE', 'LONGITUDE']].drop_duplicates().shape[0]:,}"
        ],
        'rdo_optimized': [
            f"{len(rdo_optimized):,}",
            rdo_optimized.shape[1],
            f"{rdo_optimized.memory_usage(deep=True).sum() / (1024**2):.1f}",
            f"{(rdo_optimized['DATA_OCORRENCIA_BO'].max() - rdo_optimized['DATA_OCORRENCIA_BO'].min()).days / 365.25:.1f}",
            f"{rdo_optimized[['LATITUDE', 'LONGITUDE']].drop_duplicates().shape[0]:,}"
        ],
        'Melhoria': [
            f"{(1 - len(rdo_optimized)/len(rdo_clean))*100:.1f}% menos",
            f"{(1 - rdo_optimized.shape[1]/rdo_clean.shape[1])*100:.1f}% menos",
            f"{(1 - rdo_optimized.memory_usage(deep=True).sum()/rdo_clean.memory_usage(deep=True).sum())*100:.1f}% menos",
            "mantido",
            f"{(1 - rdo_optimized[['LATITUDE', 'LONGITUDE']].drop_duplicates().shape[0] / rdo_clean[['LATITUDE', 'LONGITUDE']].drop_duplicates().shape[0])*100:.1f}% menos"
        ]
    })
    
    display(comparison)
    
    print("\n✓ Dataset otimizado pronto para pipeline de modelagem")
    print("\nUse 'rdo_optimized' nas próximas células para:")
    print("  - Agregação espacial H3")
    print("  - Criação de painel temporal")
    print("  - Feature engineering")
    print("  - Treinamento do modelo")
    
else:
    print("[ERRO] rdo_optimized está vazio")